In [1]:

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU detected. Enable GPU in Runtime settings.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:

!pip install -q transformers accelerate bitsandbytes


In [3]:

import torch
import transformers
import accelerate
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Accelerate: 1.15.0
Bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4


In [4]:
# Load the FP16 model

import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Clear unused GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Loading FP16 model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

# Memory measurements
allocated_memory = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_memory = torch.cuda.memory_reserved() / (1024 ** 3)
peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\nFP16 Model Loaded Successfully!")
print("Model:", MODEL_ID)
print(f"GPU allocated memory: {allocated_memory:.2f} GB")
print(f"GPU reserved memory: {reserved_memory:.2f} GB")
print(f"Peak GPU memory during loading: {peak_memory:.2f} GB")
print("Model device:", model_fp16.device)


Loading FP16 model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


FP16 Model Loaded Successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct
GPU allocated memory: 2.88 GB
GPU reserved memory: 2.93 GB
Peak GPU memory during loading: 2.88 GB
Model device: cuda:0


In [5]:

# Benchmark FP16 inference with 10 prompts

import time
import torch
import pandas as pd

prompts = [
    "What is artificial intelligence? Explain in simple terms.",
    "Explain the difference between machine learning and deep learning.",
    "Write a Python function to check whether a number is prime.",
    "What is overfitting in machine learning, and how can it be prevented?",
    "Explain SQL joins with a simple example.",
    "What is retrieval-augmented generation (RAG)?",
    "Explain the difference between precision and recall.",
    "What are the main benefits of using cloud computing?",
    "Write five practical applications of generative AI.",
    "Explain how a transformer model processes text."
]

def create_messages(prompt):
    return [{"role": "user", "content": prompt}]

def generate_fp16(prompt, max_new_tokens=128):
    messages = create_messages(prompt)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {k: v.to(model_fp16.device) for k, v in inputs.items()}
    input_tokens = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        output = model_fp16.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    generated_ids = output[0][input_tokens:]
    output_tokens = len(generated_ids)

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_seconds": elapsed,
        "tokens_per_second": output_tokens / elapsed if elapsed > 0 else 0,
        "answer": answer
    }

# Warm-up (not included in benchmark)
print("Running warm-up...")
_ = generate_fp16("Explain artificial intelligence briefly.")

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Run all 10 prompts
fp16_results = []

for i, prompt in enumerate(prompts, start=1):
    print(f"Running prompt {i}/10...")

    result = generate_fp16(prompt)
    result["prompt_id"] = i
    result["prompt"] = prompt
    fp16_results.append(result)

    print(
        f"Output tokens: {result['output_tokens']} | "
        f"Speed: {result['tokens_per_second']:.2f} tokens/sec"
    )

# Aggregate measurements
total_output_tokens = sum(r["output_tokens"] for r in fp16_results)
total_generation_time = sum(r["latency_seconds"] for r in fp16_results)

fp16_avg_speed = (
    total_output_tokens / total_generation_time
    if total_generation_time > 0 else 0
)

fp16_peak_memory = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

print("\n========== FP16 BENCHMARK SUMMARY ==========")
print("Prompts tested:", len(fp16_results))
print("Total output tokens:", total_output_tokens)
print(f"Total generation time: {total_generation_time:.2f} seconds")
print(f"Overall generation speed: {fp16_avg_speed:.2f} tokens/sec")
print(f"Peak GPU allocated memory: {fp16_peak_memory:.2f} GB")

# Display results
fp16_df = pd.DataFrame(fp16_results)
display(fp16_df[[
    "prompt_id",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "tokens_per_second"
]])


Running warm-up...
Running prompt 1/10...
Output tokens: 128 | Speed: 28.67 tokens/sec
Running prompt 2/10...
Output tokens: 128 | Speed: 33.76 tokens/sec
Running prompt 3/10...
Output tokens: 128 | Speed: 34.26 tokens/sec
Running prompt 4/10...
Output tokens: 128 | Speed: 29.20 tokens/sec
Running prompt 5/10...
Output tokens: 128 | Speed: 34.35 tokens/sec
Running prompt 6/10...
Output tokens: 128 | Speed: 34.54 tokens/sec
Running prompt 7/10...
Output tokens: 128 | Speed: 30.96 tokens/sec
Running prompt 8/10...
Output tokens: 128 | Speed: 31.36 tokens/sec
Running prompt 9/10...
Output tokens: 128 | Speed: 33.74 tokens/sec
Running prompt 10/10...
Output tokens: 128 | Speed: 31.41 tokens/sec

========== FP16 BENCHMARK SUMMARY ==========
Prompts tested: 10
Total output tokens: 1280
Total generation time: 39.89 seconds
Overall generation speed: 32.09 tokens/sec
Peak GPU allocated memory: 2.89 GB


,prompt_id,input_tokens,output_tokens,latency_seconds,tokens_per_second
0,1,39,128,4.465125,28.666611
1,2,40,128,3.791931,33.755886
2,3,41,128,3.736074,34.260561
3,4,45,128,4.383767,29.198632
4,5,38,128,3.726763,34.346164
5,6,40,128,3.706318,34.535623
6,7,38,128,4.133702,30.964981
7,8,39,128,4.081953,31.357537
8,9,38,128,3.793449,33.742377
9,10,38,128,4.074774,31.412788


In [6]:
# Load the same model in 4-bit quantization

import gc
import torch

del model_fp16
gc.collect()
torch.cuda.empty_cache()

print("FP16 model unloaded.")
print(
    "Current GPU memory:",
    round(torch.cuda.memory_allocated() / (1024**3), 2),
    "GB"
)


FP16 model unloaded.
Current GPU memory: 0.01 GB


In [7]:

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Reset memory statistics
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# 4-bit quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading 4-bit quantized model...")

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

allocated_memory = torch.cuda.memory_allocated() / (1024**3)
reserved_memory = torch.cuda.memory_reserved() / (1024**3)
peak_memory = torch.cuda.max_memory_allocated() / (1024**3)

print("\n4-bit Model Loaded Successfully!")
print("Model:", MODEL_ID)
print(f"GPU allocated memory: {allocated_memory:.2f} GB")
print(f"GPU reserved memory: {reserved_memory:.2f} GB")
print(f"Peak GPU memory during loading: {peak_memory:.2f} GB")


Loading 4-bit quantized model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


4-bit Model Loaded Successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct
GPU allocated memory: 1.08 GB
GPU reserved memory: 2.91 GB
Peak GPU memory during loading: 1.11 GB


In [8]:
# Benchmark 4-bit inference with the same 10 prompts

import time
import torch
import pandas as pd

def generate_4bit(prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        k: v.to(model_4bit.device)
        for k, v in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        output = model_4bit.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    generated_ids = output[0][input_tokens:]
    output_tokens = len(generated_ids)

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_seconds": elapsed,
        "tokens_per_second": (
            output_tokens / elapsed if elapsed > 0 else 0
        ),
        "answer": answer
    }


# Warm-up (excluded from benchmark)
print("Running 4-bit warm-up...")
_ = generate_4bit("Explain artificial intelligence briefly.")

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Run the same 10 prompts
results_4bit = []

for i, prompt in enumerate(prompts, start=1):
    print(f"Running prompt {i}/10...")

    result = generate_4bit(prompt)
    result["prompt_id"] = i
    result["prompt"] = prompt
    results_4bit.append(result)

    print(
        f"Output tokens: {result['output_tokens']} | "
        f"Speed: {result['tokens_per_second']:.2f} tokens/sec"
    )

# Aggregate measurements
total_output_tokens_4bit = sum(
    r["output_tokens"] for r in results_4bit
)

total_time_4bit = sum(
    r["latency_seconds"] for r in results_4bit
)

avg_speed_4bit = (
    total_output_tokens_4bit / total_time_4bit
    if total_time_4bit > 0 else 0
)

peak_memory_4bit = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

print("\n========== 4-BIT BENCHMARK SUMMARY ==========")
print("Prompts tested:", len(results_4bit))
print("Total output tokens:", total_output_tokens_4bit)
print(f"Total generation time: {total_time_4bit:.2f} seconds")
print(f"Overall generation speed: {avg_speed_4bit:.2f} tokens/sec")
print(f"Peak GPU allocated memory: {peak_memory_4bit:.2f} GB")

results_4bit_df = pd.DataFrame(results_4bit)

display(results_4bit_df[[
    "prompt_id",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "tokens_per_second"
]])


Running 4-bit warm-up...
Running prompt 1/10...
Output tokens: 103 | Speed: 16.57 tokens/sec
Running prompt 2/10...
Output tokens: 128 | Speed: 15.22 tokens/sec
Running prompt 3/10...
Output tokens: 128 | Speed: 15.17 tokens/sec
Running prompt 4/10...
Output tokens: 128 | Speed: 16.60 tokens/sec
Running prompt 5/10...
Output tokens: 128 | Speed: 10.12 tokens/sec
Running prompt 6/10...
Output tokens: 128 | Speed: 15.13 tokens/sec
Running prompt 7/10...
Output tokens: 128 | Speed: 15.09 tokens/sec
Running prompt 8/10...
Output tokens: 128 | Speed: 16.41 tokens/sec
Running prompt 9/10...
Output tokens: 128 | Speed: 15.16 tokens/sec
Running prompt 10/10...
Output tokens: 128 | Speed: 15.24 tokens/sec

========== 4-BIT BENCHMARK SUMMARY ==========
Prompts tested: 10
Total output tokens: 1255
Total generation time: 85.02 seconds
Overall generation speed: 14.76 tokens/sec
Peak GPU allocated memory: 1.09 GB


,prompt_id,input_tokens,output_tokens,latency_seconds,tokens_per_second
0,1,39,103,6.217378,16.566470
1,2,40,128,8.409428,15.221011
2,3,41,128,8.438640,15.168320
3,4,45,128,7.710024,16.601764
4,5,38,128,12.652920,10.116242
5,6,40,128,8.462553,15.125460
6,7,38,128,8.484170,15.086920
7,8,39,128,7.801540,16.407017
8,9,38,128,8.445808,15.155448
9,10,38,128,8.400030,15.238041


In [9]:
# Save benchmark results

import os
import json
import pandas as pd

base_dir = "Task3-Local-LLM-Inference"
results_dir = os.path.join(base_dir, "results")
os.makedirs(results_dir, exist_ok=True)

# Save detailed results, including generated answers
comparison_rows = []

for fp, q4 in zip(fp16_results, results_4bit):
    comparison_rows.append({
        "prompt_id": fp["prompt_id"],
        "prompt": fp["prompt"],
        "fp16_input_tokens": fp["input_tokens"],
        "fp16_output_tokens": fp["output_tokens"],
        "fp16_latency_seconds": fp["latency_seconds"],
        "fp16_tokens_per_second": fp["tokens_per_second"],
        "fp16_answer": fp["answer"],
        "4bit_input_tokens": q4["input_tokens"],
        "4bit_output_tokens": q4["output_tokens"],
        "4bit_latency_seconds": q4["latency_seconds"],
        "4bit_tokens_per_second": q4["tokens_per_second"],
        "4bit_answer": q4["answer"]
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(
    os.path.join(results_dir, "prompt_comparison.csv"),
    index=False
)

comparison_df.to_json(
    os.path.join(results_dir, "prompt_comparison.json"),
    orient="records",
    indent=4
)

# Save summary metrics
summary = {
    "model": MODEL_ID,
    "gpu": torch.cuda.get_device_name(0),
    "fp16": {
        "prompts": len(fp16_results),
        "total_output_tokens": sum(r["output_tokens"] for r in fp16_results),
        "total_generation_time_seconds": 39.89,
        "overall_tokens_per_second": 32.09,
        "peak_gpu_allocated_memory_gb": 2.89
    },
    "4bit_nf4": {
        "prompts": len(results_4bit),
        "total_output_tokens": sum(r["output_tokens"] for r in results_4bit),
        "total_generation_time_seconds": 85.02,
        "overall_tokens_per_second": 14.76,
        "peak_gpu_allocated_memory_gb": 1.09,
        "quantization": "NF4",
        "double_quantization": True,
        "compute_dtype": "float16"
    }
}

with open(os.path.join(results_dir, "metrics_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Saved benchmark files:")
print(os.listdir(results_dir))


Saved benchmark files:
['prompt_comparison.csv', 'metrics_summary.json', 'prompt_comparison.json']


In [10]:

import os
import shutil

base_dir = "Task3-Local-LLM-Inference"
results_dir = os.path.join(base_dir, "results")
os.makedirs(results_dir, exist_ok=True)

# Confirm all expected result files exist
expected_files = [
    "prompt_comparison.csv",
    "prompt_comparison.json",
    "metrics_summary.json"
]

for filename in expected_files:
    path = os.path.join(results_dir, filename)
    assert os.path.exists(path), f"Missing file: {filename}"

# Create ZIP archive
zip_path = shutil.make_archive(
    "/content/Task3_Results",
    "zip",
    root_dir=base_dir,
    base_dir="results"
)

print("ZIP created:", zip_path)
print("Files:", os.listdir(results_dir))


ZIP created: /content/Task3_Results.zip
Files: ['prompt_comparison.csv', 'metrics_summary.json', 'prompt_comparison.json']


In [11]:

from google.colab import files

files.download("/content/Task3_Results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>